# Step 1 — immutable Kaggle T4 x2 run

This notebook is only a non-interactive launcher. The pinned SHA must name the source commit containing this runner.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/escher-bach/actuallybuildingstuff.git'
# Replace this placeholder with the full SHA of the commit containing this runner.
GIT_COMMIT = '218c1d0fa265c0754ebc42caece1b5c3f371acab'
CONFIG_REL = 'step1/configs/kaggle/t4x2_preflight.toml'

WORKING = Path('/kaggle/working')
SOURCE = WORKING / 'actuallybuildingstuff'
PROJECT = SOURCE / 'baby-llm-foundations'
OUTPUT = WORKING / 'step1-results'
assert len(GIT_COMMIT) == 40 and all(c in '0123456789abcdef' for c in GIT_COMMIT)
assert not SOURCE.exists(), f'fresh batch session required; already exists: {SOURCE}'
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
env = os.environ.copy()
env.update({'GIT_TERMINAL_PROMPT': '0', 'PYTHONUNBUFFERED': '1', 'PIP_DISABLE_PIP_VERSION_CHECK': '1', 'WANDB_MODE': 'disabled', 'TOKENIZERS_PARALLELISM': 'false'})
subprocess.run(['git', 'clone', REPO_URL, str(SOURCE)], check=True, env=env)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', GIT_COMMIT], check=True, env=env)
resolved = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True, env=env).strip()
assert resolved == GIT_COMMIT, (resolved, GIT_COMMIT)
assert (PROJECT / CONFIG_REL).is_file(), PROJECT / CONFIG_REL


In [ ]:
cmd = [sys.executable, '-m', 'step1_experiments.runner', '--config', str(PROJECT / CONFIG_REL), '--output-root', str(OUTPUT), '--resume', 'auto']
completed = subprocess.run(cmd, cwd=str(PROJECT / 'step1' / 'python'), env=env, check=False)
if completed.returncode != 0:
    raise RuntimeError(f'Step 1 runner failed with exit code {completed.returncode}; download the failure bundle from {OUTPUT}')
